# **BONUS MATERIAL: Access and analyze pre-calculated climate indicators for an ensemble of climate simulations**

This notebook shows how to create additional figures from the indicator data generated in `tutorial-indicators.ipynb`.

First re-run **Steps 1 to 7** in `tutorial-indicators.ipynb` using the input settings specified for each example below. Each new combination of inputs will generate a new dataset and will save it as a `.zarr` file in the output folder.

Once the new datasets have been created and saved, run the code provided in each section to generate the figures.

### **Preparation – Imports & configuration**

Run this cell once at the very beginning of the notebook to load the Python packages. No modification needed

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import panel as pn
import xarray as xr
from xscen import ensembles as xens

import hvplot.pandas
import hvplot.xarray

warnings.simplefilter("ignore")
pn.extension("tabulator")

### **1 - Choropleth maps: Add indicator values to the original polygon layer**

The first example compares the ensemble median winter (DJF) total precipitation (`prtot`) for the baseline period (1991–2020) and future period (2071–2100) under SSP3-7.0 using interactive choropleth maps.
A choropleth map is a map in which each geographic region is shaded according to the value of a variable, making it easy to visualize spatial patterns across polygons.

To generate the maps:

1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7** but update the **Step 1** inputs as follows:
- `spatial_avg = True`
- `time_avg = True`
- `ensemble_percentiles = True`

    This will generate and save a dataset of spatially averaged, 30-year mean values with ensemble percentiles.

2. Open the newly generated output dataset (`*ensemble_percentiles_30yAvg_spatial-avg*.zarr`) and use it to:
* Add data columns to the original polygon layer
* Export the updated polygon layer to GeoJSON (or shapefile)
* Create interactive maps in the PAVICS environment

First, specify the data source, polygon file and temporal group.
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>

In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
data_source = "CanDCS-M6"                                          # data source

# region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"         # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

temp_group = "seasonal"                                            # which dataset to access: one of "seasonal", "monthly", "annual" etc.

Run the cell below add data columns to the original polygon layer and export it to GeoJSON.

In [ ]:
# Where to save Zarr and CSV files? 
output_folder = "output/" + Path(polygon_file).stem

# convert to Path
polygon_file = Path(polygon_file)
output_folder = Path(output_folder)
output_folder.mkdir(parents=True, exist_ok=True)

# export only the ensemble median : could add [10, 50, 90]
keep_perc = [50]

# find appropriate output data
zarr = list(output_folder.joinpath(data_source, 'zarr').glob(f'{temp_group}_*ensemble_percentiles_30yAvg_spatial-avg_{polygon_file.stem}.zarr'))
if len(zarr) !=1:
    print(f'found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above.')
else:
    zarr = zarr[0]
    gdf = gpd.read_file(polygon_file).to_crs(epsg=4326)
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    
    if 'realization' in ds1.dims:
        ds1 = xens.ensemble_stats(
            ds1,
            {"ensemble_percentiles": {"split": False}},
        )
   
    #define output shp file path
    outpoly = output_folder.joinpath(data_source, 'geojson', f"{zarr.stem.replace('_members','_ensemble_percentiles')}.geojson")
    # create parent directory
    outpoly.parent.mkdir(exist_ok=True, parents=True)
    
    # concatenate data as shapefile columns
    for vv in ds1.data_vars:
        for ssp in ds1.scenario.values:
            for seas in ds1.season.values:
                for hh in ds1.horizon.values:
                    for perc in keep_perc:
                        # define column name
                        vvv = f"{vv}-{seas}-{hh}-{ssp}_p{perc}"
                        # get data values
                        data = ds1[vv].sel(season=seas, scenario=ssp, percentiles=perc, time=(ds1.horizon == hh)).squeeze()
                        # get units as well
                        units = data.attrs['units']
                        # convert to pandas df
                        data = data.to_dataframe()
                        # append new data column to polygon layer
                        gdf[f"{vvv} ({units})"] = data[vv]
    #export to geojson
    gdf.to_file(outpoly, driver="GeoJSON")      

    print(f"Updated polygon layer exported to: {outpoly}")


Let's take a look at the output. The output is a GeoJSON file with the same geometry as the input polygon file, but with additional columns for each variable, season, horizon, scenario, and percentile combination.

In [ ]:
print(outpoly)
gdf = gpd.read_file(outpoly)
gdf.head()

Run the cell below to plot the ensemble median of average winter total precipitation for sub-regions for baseline (1991–2020) and future (2071–2100) periods under SSP3-7.0. By comparing the two maps, you can see how precipitation levels vary across regions and how they are projected to change over time.

In [ ]:
# baseline period
h1 = '1991-2020'
# future period 
h2 = '2071-2100'

# variable for baseline period
var1 = f'prtot-DJF-{h1}-ssp370_p50 (mm)'
# variable for future period
var2 = f'prtot-DJF-{h2}-ssp370_p50 (mm)'

# set color limits (min and max of our two variables)
clim = (gdf[[var1, var2]].min().min(), gdf[[var1, var2]].max().max())

# plot interactive map
gdf.hvplot(c=var1, clim=clim, geo=True, tiles='CartoLight', alpha=0.65, title=h1, cmap='spectral') \
+ gdf.hvplot(c=var2, clim=clim, geo=True, tiles='CartoLight', alpha=0.65, title=h2, cmap='spectral')

### **2 - Maps of gridded data for our region**

The second example shows how to visualize pre-calculated indicators as gridded maps for a selected region, compare a reference period with other time horizons, and export the results as GeoTIFF files for use in GIS.

To generate the maps:

1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7** but update the **Step 1** inputs as follows:
- `spatial_avg = False`
- `time_avg = True`
- `ensemble_percentiles = True`

    This will generate and save a dataset of gridded 30-year mean values with ensemble percentiles.

2. Open the newly generated output dataset (`*ensemble_percentiles_30yAvg.zarr`) and use it to:
* Create interactive plots in the PAVICS environment
* Create a simple time-slider widget to navigate across horizons
* Export gridded outputs to GeoTIFF for GIS users

First, specify the temporal group, season, variable, baseline period, scenario and percentile to plot.
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>

In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
temp_group = "seasonal"                                            # which dataset to access: one of "seasonal", "monthly", "annual" etc.
season = "MAM"                                                     # select one of the four seasons: "DJF", "MAM", "JJA", "SON" 
analyze_variable = "prtot"                                         # which variable to plot
h1 = '1991-2020'                                                   # select the baseline period
scenario = 'ssp370'                                                # select the emissions scenario
keep_perc = [50]                                                   # which percentiles to keep (e.g., 10, 50, 90). [50] keeps only the ensemble median.

Run the cell below to create the interactive maps with a slider to explore other horizons.

In [ ]:
# find appropriate output data
zarr = list(output_folder.joinpath(data_source, 'zarr').glob(f'{temp_group}_*ensemble_percentiles_30yAvg.zarr'))
if len(zarr) !=1:
    print(f'found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above.')
else:
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    ds_data = ds1.sel(scenario=scenario, season=season, percentiles=keep_perc).squeeze()

    xlim = [ds_data.lon.min().values - 0.5, ds_data.lon.max().values + 0.5]
    ylim = [ds_data.lat.min().values - 0.5, ds_data.lat.max().values + 0.5]
    # define color limits for all time periods
    clim = (ds_data[analyze_variable].min().values, ds_data[analyze_variable].max().values)

    # define common keyword args for both reference and future maps
    kwargs = dict(xlim=xlim, ylim=ylim, clim=clim, geo=True, tiles='CartoLight', alpha=0.65, cmap='spectral', attr_labels=False, framewise=False)

    # reference data plot
    ref_plot = ds_data[analyze_variable].sel(time=ds_data.horizon == h1).squeeze().hvplot.image(title=h1, **kwargs)

    # create a time-slider with dataset horizons as options
    time_opt_dict = {f"{ds_data.sel(time=tt).horizon.values}":tt for tt in ds_data.time.values}
    time_slider = pn.widgets.DiscreteSlider(name='Time', value=ds_data.time.values[-1], options=time_opt_dict)

    # future selection bound to time slider widget
    fut_plot = ds_data[analyze_variable].interactive(loc='bottom').sel(time=time_slider).hvplot.image(**kwargs)

    # plot side by side (+)
    plt1 = ref_plot + fut_plot

    # create a title using metadata and season choice
    md_title = pn.pane.Markdown(f"## {ds_data[analyze_variable].attrs['long_name'].rstrip('.').capitalize()} ({ds_data[analyze_variable].attrs['units']})<br>Season : {season}<br>Emissions : {scenario.upper()}")

    # display title and plot in the jupyterlab
    display(pn.Column(md_title, plt1))

### **Exporting gridded outputs to GeoTIFF**

The code below loops over all non-spatial dimensions in the dataset and exports each 2D slice as a separate GeoTIFF file. These files can then be opened in standard GIS software.

To download them from the PAVICS environment, right-click the `geotiff` subfolder in the output directory and select "Download as an archive" to quickly save all layers locally.

In [ ]:
# find appropriate output data
zarr = list(output_folder.joinpath(data_source, 'zarr').glob(f'{temp_group}_*ensemble_percentiles_30yAvg.zarr'))
if len(zarr) !=1:
    print(f'found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above.')
else:
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    # loop over data variables
    for vv in ds1.data_vars:
        print(f"Exporting {vv} to geotiffs ... ")
        # loop over non-spatial dimensions
        for tt in ds1.time:
            for pp in keep_perc:
                for ss in ds1.scenario:
                    for seas in ds1.season:
                        #select 2D data slice
                        data = ds1[vv].sel(time=tt, percentiles=pp, scenario=ss, season=seas)#.squeeze()
                        
                        # create output file path
                        outtiff = Path(output_folder / data_source / "geotiff" / vv /
                                       f"{zarr.stem}_{ss.values}_{vv}({data.attrs['units']})_{data.horizon.values}_{seas.values}_p{pp}.tiff")
                        
                        # creat parent folder 
                        outtiff.parent.mkdir(parents=True, exist_ok=True)
                        
                        # Geotiff anchor point is top-left corner
                        # sort data w/ lat ascending=False 
                        # sort data w/ lon ascending=True
                        data = data.sortby(['lat'],ascending=False).sortby(['lon'], ascending=True).squeeze().rio.set_crs("epsg:4326").rio.set_spatial_dims(
                            x_dim="lon", y_dim="lat"
                        )
                        
                        #write data to file
                        data.rio.to_raster(outtiff)
                        
        print('Done. Files saved to: ' + str(outtiff.parent))      